# api_dashboard_manifest_gate_v1

验证 paper/sim API intent 是否把 dashboard export 的 `manifest_source` 纳入 readiness 和 intent metadata。目标是避免 paper execution 直接消费缺失 manifest 或 legacy `fixed_sidecars` 来源的 dashboard。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

EXP = "api_dashboard_manifest_gate_v1"
ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.api import build_target_position_intent_from_dashboard

WORKDIR = ROOT / "research" / EXP
RUNTIME = ROOT / "runtime" / "research" / EXP
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def base_dashboard() -> dict:
    return {
        "schema_version": "dashboard.model_matrix.v1",
        "artifact_id": "unit_dashboard",
        "top_models": [
            {
                "rank": 1,
                "model": "ridge_mse__all_features",
                "base_model": "ridge_mse",
                "feature_set": "all_features",
                "avg_position": 0.5,
                "stability_score": 4.0,
                "max_drawdown_pct": 8.0,
            }
        ],
    }


In [2]:
cases = {}

strong = base_dashboard()
strong["manifest_source"] = {
    "kind": "analysis_manifest",
    "path": "matrix_card/analysis_manifest.json",
    "manifest_hash": "a" * 64,
    "producer": "src.analysis.exports",
}
cases["strong_analysis_manifest"] = strong

missing = base_dashboard()
cases["missing_manifest_source"] = missing

fixed = base_dashboard()
fixed["manifest_source"] = {
    "kind": "fixed_sidecars",
    "path": "",
    "manifest_hash": "",
    "producer": "src.dashboard.model_matrix",
}
cases["fixed_sidecars_manifest_source"] = fixed

results = []
for case_name, payload in cases.items():
    intent = build_target_position_intent_from_dashboard(
        payload,
        instrument="SOXL",
        min_position=0.1,
        max_position=0.75,
    )
    meta = intent.metadata or {}
    results.append(
        {
            "case": case_name,
            "target_position": intent.target_position,
            "execution_gate": meta.get("execution_gate", ""),
            "dashboard_manifest_status": meta.get("dashboard_manifest_status", ""),
            "dashboard_manifest_kind": meta.get("dashboard_manifest_kind", ""),
            "dashboard_manifest_hash_present": bool(meta.get("dashboard_manifest_hash", "")),
            "readiness_overview_status": meta.get("readiness_overview_status", ""),
            "readiness_overview_block_gates": meta.get("readiness_overview_block_gates", ""),
        }
    )

by_case = {row["case"]: row for row in results}
assert by_case["strong_analysis_manifest"]["execution_gate"] == "paper_review"
assert by_case["strong_analysis_manifest"]["dashboard_manifest_status"] == "complete"
assert by_case["strong_analysis_manifest"]["target_position"] == 0.5
assert by_case["missing_manifest_source"]["execution_gate"] == "blocked_to_min_position"
assert by_case["missing_manifest_source"]["dashboard_manifest_status"] == "missing"
assert by_case["missing_manifest_source"]["target_position"] == 0.1
assert by_case["fixed_sidecars_manifest_source"]["execution_gate"] == "blocked_to_min_position"
assert by_case["fixed_sidecars_manifest_source"]["dashboard_manifest_status"] == "weak_kind"
assert by_case["fixed_sidecars_manifest_source"]["target_position"] == 0.1

summary = {
    "schema_version": "research.api_dashboard_manifest_gate_v1",
    "experiment": EXP,
    "ready_case_count": sum(1 for row in results if row["execution_gate"] == "paper_review"),
    "blocked_case_count": sum(1 for row in results if row["execution_gate"] == "blocked_to_min_position"),
    "cases": results,
}
write_json(WORKDIR / "analysis_summary.json", summary)
summary


{'schema_version': 'research.api_dashboard_manifest_gate_v1',
 'experiment': 'api_dashboard_manifest_gate_v1',
 'ready_case_count': 1,
 'blocked_case_count': 2,
 'cases': [{'case': 'strong_analysis_manifest',
   'target_position': 0.5,
   'execution_gate': 'paper_review',
   'dashboard_manifest_status': 'complete',
   'dashboard_manifest_kind': 'analysis_manifest',
   'dashboard_manifest_hash_present': True,
   'readiness_overview_status': 'ready',
   'readiness_overview_block_gates': ''},
  {'case': 'missing_manifest_source',
   'target_position': 0.1,
   'execution_gate': 'blocked_to_min_position',
   'dashboard_manifest_status': 'missing',
   'dashboard_manifest_kind': '',
   'dashboard_manifest_hash_present': False,
   'readiness_overview_status': 'blocked',
   'readiness_overview_block_gates': 'dashboard_manifest'},
  {'case': 'fixed_sidecars_manifest_source',
   'target_position': 0.1,
   'execution_gate': 'blocked_to_min_position',
   'dashboard_manifest_status': 'weak_kind',
  

## 中文结论

- 做了什么：验证 `src/api` 生成 paper/sim execution intent 时，会读取 dashboard export 的 `manifest_source`，并写入 intent metadata/readiness。
- 核心指标：3 个场景中，只有 `analysis_manifest` 且 `path`、`manifest_hash`、`producer` 齐全的 dashboard 保持 `paper_review` 和 target_position=0.5；缺失 manifest 来源和 `fixed_sidecars` 来源都会进入 `blocked_to_min_position`，target_position=0.1。
- regime 差异：本次是 API/readiness 契约验证，不直接比较市场 regime；它保证后续 regime 分层和模型差异分析必须经 manifest 化 dashboard 后才能影响 paper intent。
- 是否晋升：该门禁应进入 `src/api` 与 `src/schedule` 的 paper execution 链路；没有新增 production strategy。
- 风险：当前只验证 dashboard export 的 manifest 来源字段，不递归校验 manifest 内每个 public artifact 的业务内容。
- 下一步：继续把 production health、paper execution、dashboard page 的 readiness 展示统一到同一组 manifest/source gate。